# Pluralistic Leaderboards -- reproduction walkthrough

This notebook reproduces the LMArena experiment from Section 5.2 of
Haghtalab, Procaccia, Shao, Wang, & Yang (Jan 2026), [Pluralistic Leaderboards](https://procaccia.info/wp-content/uploads/2026/01/leaderboard.pdf).

Pipeline:
  1. Load the LMArena 140k dataset and filter to top m=20 LLMs.
  2. Compute per-category Bradley-Terry rankings -> Mallows central rankings.
  3. Build a Mallows mixture and sample synthetic user rankings.
  4. Run BT, Algorithm 2, Algorithm 3 and compare gamma_hat stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pluralistic_leaderboards.algorithms import (
    bradley_terry_ranking,
    make_mallows_mixture_oracle,
    ranking_via_committee_monotonicity,
    ranking_via_geometric_checkpoints,
)
from pluralistic_leaderboards.data import load_lmarena, per_category_central_rankings
from pluralistic_leaderboards.evaluation import stability_curve

In [ ]:
sample = load_lmarena(n_models=20, max_battles=20_000, seed=42)
print(f"loaded {sample.n_battles} battles across {sample.n_models} models")
print(f"categories ({len(sample.category_to_idx)}):", list(sample.category_to_idx.keys()))
print("mixture weights:", sample.category_mixture_weights.round(3).tolist())

In [ ]:
central_rankings = per_category_central_rankings(sample)
centers = [central_rankings[c] for c in sample.category_to_idx.keys()]
weights = sample.category_mixture_weights
for cat, c in zip(sample.category_to_idx.keys(), centers):
    print(f"  [{cat}] top: {sample.model_names[int(c[0])]}")

In [ ]:
phi = 0.5
rng = np.random.default_rng(42)
oracle = make_mallows_mixture_oracle(centers=centers, phi=phi, weights=weights)
bt = bradley_terry_ranking(sample.battles, m=sample.n_models)
a2 = ranking_via_geometric_checkpoints(oracle, m=sample.n_models, rng=rng, growth_factor=2.0, epsilon=0.1, n_users_per_round=300, n_lottery_samples_L=15, n_committee_eval=80, n_mwu_iters=12)
a3 = ranking_via_committee_monotonicity(oracle, m=sample.n_models, rng=rng, epsilon=0.1, n_users_per_round=300, n_lottery_samples_L=15, n_committee_eval=80, n_mwu_iters=12)
for name, r in [("BT", bt), ("A2", a2), ("A3", a3)]:
    print(name, [sample.model_names[i] for i in r[:5]])

In [ ]:
eval_users = oracle(10_000, np.random.default_rng(99))
bt_curve = stability_curve(eval_users, bt, m=sample.n_models)
a2_curve = stability_curve(eval_users, a2, m=sample.n_models)
a3_curve = stability_curve(eval_users, a3, m=sample.n_models)
fig, ax = plt.subplots(figsize=(7, 4))
k_axis = np.arange(1, len(bt_curve) + 1)
ax.plot(k_axis, bt_curve, "o-", label="Bradley-Terry")
ax.plot(k_axis, a2_curve, "v-", label="Algorithm 2")
ax.plot(k_axis, a3_curve, "x-", label="Algorithm 3")
ax.axhline(1.0, ls="--", color="gray", alpha=0.6)
ax.set_xlabel("top-k committee size")
ax.set_ylabel(r"stability ratio $\hat\gamma$")
ax.set_title(f"LMArena stability curves (phi = {phi})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()